**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/encorescience/encore-registry", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/jones-2021"], check=True)
    os.chdir("/content/encore/entries/jones-2021")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Aery Jones et al. (2021) from dandiset 000165: CSD during SWRs in cohort 1

**Paper.** Aery Jones EA, Rao A, Zilberter M, Djukic B, Bant JS, Gillespie AK, Koutsodendris N, Nelson M,
Yoon SY, Huang K, Yuan H, Gill TM, Huang Y, Frank LM (2021). Dentate gyrus and CA3 GABAergic
interneurons bidirectionally modulate signatures of internal and external drive to CA1. *Cell
Reports* 37, 110159. [doi:10.1016/j.celrep.2021.110159](https://doi.org/10.1016/j.celrep.2021.110159).
The claims file (`claims/jones-2021.yaml`) was written from the accepted author manuscript in PubMed
Central (PMC9069800) with its main supplement; Table S7 was not available to the reader.

**Dandiset.** [DANDI:000165](https://dandiarchive.org/dandiset/000165/0.211118.1526), version
0.211118.1526 (doi:10.48324/dandi.000165/0.211118.1526). The paper links the draft, not a version.
This is the only published version, and the draft has the same 572 assets and size, so the most
recent published version is pinned. Files are streamed with remfile and h5py; nothing is written to disk.

**Notebooks.** `rest_home_cage.ipynb` tests the claims on the home cage (rest) sessions;
`run_linear_track.ipynb` tests those on the linear track (run) sessions; `csd_cohort1.ipynb` tests
the CSD claim on the single-shank home cage sessions of cohort 1. Methods are in the helper
`jones2021.py`, whose docstring says where each parameter comes from.

**This notebook** tests one claim, fig2ef-csd-sr, on the home cage sessions of cohort 1 (single-shank
probe, 32 sites at 50 um). The other claims are tested in the two other notebooks; their table of
claims lists every claim and where it is tested.

**Method.** SWRs as in the rest notebook (3 SD on the "ca1-pyr 1" site, during extended immobility,
edge and artifact rules). CSD: each site smoothed with its neighbours as (a + 2b + c)/4 (code RTCSD.m,
as the criterion says), then minus the second difference across depth over the spacing squared, so that
sources are positive and sinks negative. Per SWR, the peak source and the peak sink (as a magnitude)
over the CA1-sr sites in the 100 ms after onset (Methods, Analysis of neural data, paragraph 4). The
LMM and genotype test are Method M as in the other notebooks; Holm over the 24 rest comparisons,
with the comparisons computed in the rest notebook not repeated here, so p < 0.05/24 suffices.


In [ ]:
import time
import warnings

import numpy as np
import pandas as pd

import jones2021 as J

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 220)
started = time.time()
assets = J.list_assets()
hc = assets[assets.kind == "HomeCage"]
first = hc.drop_duplicates("subject")        # the probe is a property of the mouse: one file per mouse is read
probe = first.merge(J.scan(first.asset_id.tolist(), workers=16), on="asset_id").set_index("subject").probe
c1 = hc[hc.subject.map(probe).str.contains("A1x32")]
c1 = c1.merge(J.scan(c1.asset_id.tolist(), workers=16), on="asset_id")
print("Probes by mouse:", probe.value_counts().to_dict())
print(f"dandiset {J.DANDISET_ID} version {J.DANDISET_VERSION}: {len(c1)} single-shank home cage files of {len(hc)}")
t0 = time.time()
results = J.process_many(c1.asset_id.tolist(), "CSD", workers=4)
errors = [r for r in results if "error" in r]
print(f"processed in {(time.time() - t0) / 60:.1f} min; failed: {len(errors)}")
for e in errors:
    print("  ", e)


## Data check

In [ ]:
S = J.session_table(results)
print("Probes:", c1.probe.unique())
print(S.groupby(["group", "treatment"]).agg(sessions=("session", "size"), mice=("animal", "nunique"),
                                             minutes=("minutes", "median")).round(1).to_string())
seq = S.assign(n=S.session.str.extract(r"(\d+)")[0].astype(int)).sort_values(["animal", "n"]).groupby("animal").agg(
    group=("group", "first"), sequence=("treatment", lambda t: "".join("C" if x == "CNO" else "v" for x in t)))
print("\nTreatment sequence per mouse (v vehicle, C CNO):"); print(seq.to_string())
print("Sessions skipped:", S.skipped.notna().sum())
chk = pd.DataFrame([dict(session=r["meta"]["session_id"], depth_sorted=r["depth_sorted"], n_sr=r["n_sr"],
                         rest_min=r["rest_s"] / 60, swr=len(r.get("swr", [])), seconds=r.get("seconds"),
                         maxrss_gb=r.get("maxrss_kb", 0) / 1e6) for r in results if "error" not in r])
print("\nSite depths increase with site index in every file:", bool(chk.depth_sorted.all()))
print(chk.drop(columns=["session", "depth_sorted"]).describe().round(2).T.to_string())
sw = pd.concat([results[i]["swr"].assign(group=g, treatment=t) for i, g, t in zip(S.i, S.group, S.treatment) if "swr" in results[i]])
print("\nPeak source and sink (file units per mm2) over SWRs, by group and treatment:")
print(sw.groupby(["group", "treatment"])[["source", "sink"]].describe().round(0).T.to_string())


## fig2ef-csd-sr

> During SWRs, current source density sources and sinks in CA1 stratum radiatum increase when either PV+ or SST+ interneurons are suppressed, with no difference between genotypes.

Readings (a) log when right skewed, (b) log always (Method M); the code's 5 SD SWRs are a variant.

In [ ]:
FEAT = {"source": J._col("swr", "source"), "sink": J._col("swr", "sink"),
        "source_5sd": J._sel("swr", "source", min_size=5), "sink_5sd": J._sel("swr", "sink", min_size=5)}
S = S[S.skipped.isna()].reset_index(drop=True)
TAB = J.build_tables(results, S, FEAT)
res = J.test_all(TAB, groups=("PV", "SST"), pairs=(("PV", "SST"),))
res = J.add_decisions(res, J.REST_FAMILY, ["source", "sink"], {"source_5sd": "source", "sink_5sd": "sink"})
parts = [("source", "PV", "up"), ("source", "SST", "up"), ("sink", "PV", "up"), ("sink", "SST", "up"),
         ("source", "PV vs SST", "ns"), ("sink", "PV vs SST", "ns")]
print(J.show_parts(res, parts).to_string(index=False, float_format=lambda v: f"{v:.3g}"))
print("\n5 SD variant:")
print(J.show_parts(res, [(f + "_5sd", g, e) for f, g, e in parts]).to_string(index=False, float_format=lambda v: f"{v:.3g}"))
m = TAB["sink"].groupby(["group", "animal", "cno"]).value.mean().unstack()
s_ = TAB["source"].groupby(["group", "animal", "cno"]).value.mean().unstack()
print("\nPer mouse, mean peak source and sink by treatment (0 vehicle, 1 CNO):")
print(pd.concat({"source": s_, "sink": m}, axis=1).round(0).to_string())


**Result: not reproduced.** With PV+ suppression the peak CA1-sr source (+6540 file units per mm2, p = 4.4e-11) and sink (+16200, p = 0.0014) increase, and the genotype tests are not significant (p = 0.65 and 0.23), as claimed. With SST+ suppression both rise in every mouse but not significantly (source p = 0.0078, sink p = 0.015; with the rest notebook's SST-Cre p-values the Holm family stops at its second rank, 0.05/23). Two of the four treatment effects hold, and the partial rule needs three. Reading (b) of the log transform gives the same decisions; with the code's 5 SD SWRs, reading (b) makes the SST-Cre source significant (p = 0.0012), three of four. 5 PV-Cre and 4 SST-Cre mice.


## Summary

| Claim | Outcome |
|---|---|
| fig2ef-csd-sr | not reproduced |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the differences from the paper's method.

How the analysis differs from the paper's: SWR boundaries where the envelope crosses its mean (as in the rest notebook); the CSD spacing taken as 50 um from the probe name, with values in file units per mm2; Holm decided with the rest notebook's p-values, not recomputed here.

What was found about the dataset: the cohort 1 files give site depths (rel_y) in site order and label the layers, so the CA1-sr sites can be found; cohort 1 alternates vehicle and CNO, ending with two CNO sessions in most mice.


In [ ]:
print(f"notebook ran in {(time.time() - started) / 60:.1f} min")